# Notebook 03 — Phase 2: HMM Training

Train a Hidden Markov Model on historical landslide records to:
- Identify landslide **type** (Shallow / Deep / Debris Flow / Rockfall)
- Compute **occurrence probability**

**Algorithm**: Baum-Welch (EM) via hmmlearn's MultinomialHMM  
**Data source**: `data set (1).xlsx` — 54 historical landslide events

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
import config

## 1. Load & Inspect Excel Data

In [ ]:
from phase2_hmm.data_preprocessing import load_excel_data

df = load_excel_data(config.EXCEL_FILE)
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
print(df.info())
df.describe(include='all')

## 2. Clean and Encode

In [ ]:
from phase2_hmm.data_preprocessing import clean_and_encode

df_clean, dis_enc, loc_enc = clean_and_encode(df)

# Show before/after for key columns
cols_to_show = [c for c in ['location_clean','year_int','duration_days','disaster_clean','disaster_code'] 
                if c in df_clean.columns]
df_clean[cols_to_show].head(15)

In [ ]:
import matplotlib.pyplot as plt

# Distribution of disaster types
df_clean['disaster_clean'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Disaster Type Distribution')
plt.xlabel('Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Build Observation Sequences

In [ ]:
from phase2_hmm.data_preprocessing import build_observation_sequences

obs_sequences, lengths = build_observation_sequences(df_clean)

# Sanity checks
import numpy as np
assert all(len(s) > 0 for s in obs_sequences), 'Empty sequences found!'
assert sum(lengths) == len(np.concatenate(obs_sequences)), 'Length mismatch!'

print(f'Number of sequences: {len(obs_sequences)}')
print(f'Total observations : {sum(lengths)}')

# Histogram of sequence lengths
plt.hist(lengths, bins=range(1, max(lengths)+2), color='steelblue', edgecolor='white', align='left')
plt.title('Sequence Length Distribution')
plt.xlabel('Sequence Length')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Build & Train HMM

In [ ]:
from phase2_hmm.hmm_model import build_hmm, train_hmm, print_model_parameters

n_symbols = len(dis_enc.classes_)
print(f'Observation symbols (disaster types): {n_symbols}')

hmm_model = build_hmm(
    n_components=config.HMM_N_COMPONENTS,
    n_symbols=n_symbols,
    n_iter=config.HMM_N_ITER,
)

hmm_model = train_hmm(hmm_model, obs_sequences, lengths)

## 5. Inspect Trained Parameters

In [ ]:
print_model_parameters(hmm_model)

# Validate: each row of transition matrix must sum to ~1
row_sums = hmm_model.transmat_.sum(axis=1)
assert np.allclose(row_sums, 1.0, atol=1e-6), f'Transition matrix row sums: {row_sums}'
print('Transition matrix row sums OK:', row_sums)

## 6. Visualise Transition & Emission Matrices

In [ ]:
from phase2_hmm.visualize import plot_transition_matrix, plot_emission_probabilities

state_labels = [config.HMM_STATE_LABELS[i] for i in range(config.HMM_N_COMPONENTS)]
obs_labels = list(dis_enc.classes_)

plot_transition_matrix(hmm_model.transmat_, state_labels,
                       save_path=os.path.join(config.PLOTS_DIR, 'hmm_transition.png'))

plot_emission_probabilities(hmm_model.emissionprob_, state_labels, obs_labels,
                            save_path=os.path.join(config.PLOTS_DIR, 'hmm_emission.png'))

## 7. Save Model & Encoder

In [ ]:
from phase2_hmm.hmm_model import save_hmm_model, save_encoder

save_hmm_model(hmm_model, config.HMM_MODEL_PATH)
save_encoder(dis_enc, config.HMM_ENCODER_PATH)
print('HMM model and encoder saved!')